In [3]:
import numpy as np
import xarray as xr

In [4]:
original_box_size = 7.5
new_box_size = 30
varPrefixes = ["SWCF","LWCF","PRECT"]


In [6]:
def get_weights_names(boxSize):
    """
    Create a array containing the names of the metrics weights
    
    :param boxSize: Size of the boxes over which the global data is averaged
    """

    weightsNames = []


    for i in range(1,(int(180/boxSize)+1)):
        for j in range(1,int(360/boxSize)+1):
            weightsNames.append(f"numb_{i}_{j}")

    weightsNames = np.array(weightsNames)

    return weightsNames

In [7]:
import numpy as np
import xarray as xr

def coarsen_spatial_features(dataset, old_boxsize, new_boxsize, varPrefixes):
    
    old_x_len = int(360 / old_boxsize)  
    old_y_len = int(180 / old_boxsize)  
    
        
    factor = int(new_boxsize / old_boxsize) 
    new_x_len = int(old_x_len / factor)     
    new_y_len = int(old_y_len / factor)     
    
    dataset_coarse = dataset.copy()
    vars_to_drop = []

    old_weights_names = get_weights_names(old_boxsize)
    new_weights_names = get_weights_names(new_boxsize)

    vars_to_drop.extend(old_weights_names)

    metricsWeights = dataset[old_weights_names].to_array(dim="variable")
    weights_dimensions = metricsWeights.dims[1:]
    metricsWeights = metricsWeights.values

    w_2d = metricsWeights.reshape(old_y_len, old_x_len, *metricsWeights.shape[1:])
    w_4d = w_2d.reshape(new_y_len, factor, new_x_len, factor, *metricsWeights.shape[1:])
    w_sum = w_4d.sum(axis=(1, 3))
    
    w_sum_flat = w_sum.reshape(new_y_len * new_x_len, *metricsWeights.shape[1:])
    dataset_coarse = dataset_coarse.drop_vars(old_weights_names,errors="ignore")

    for i, new_var in enumerate(new_weights_names):
        dataset_coarse[new_var] = (weights_dimensions, w_sum_flat[i])


    
    for prefix in varPrefixes:
        var_names = [f"{prefix}_{y}_{x}" for y in range(1,old_y_len+1) for x in range(1,old_x_len+1)]
        valid_vars = [v for v in var_names if v in dataset.data_vars]
        print(var_names)
        if not valid_vars:
            continue
            

        
        da = dataset[valid_vars].to_array(dim="variable")
        arr = da.values
        
        other_dims = da.dims[1:]       
        other_shapes = arr.shape[1:]  

        arr_3d = arr.reshape(old_y_len, old_x_len, *other_shapes)
        arr_5d = arr_3d.reshape(new_y_len, factor, new_x_len, factor, *other_shapes)
        
        
        w_broadcast = w_4d.reshape(*w_4d.shape, *(1,) * len(other_shapes))
        w_sum_broadcast = w_sum.reshape(*w_sum.shape, *(1,) * len(other_shapes))
        
        weighted_sum = (arr_5d * w_broadcast).sum(axis=(1, 3))
        arr_coarse = weighted_sum / w_sum_broadcast
        dataset_coarse = dataset_coarse.drop_vars(valid_vars, errors="ignore")

        for new_y in range(1,new_y_len+1):
            for new_x in range(1,new_x_len+1):
                new_var = f"{prefix}_{new_y}_{new_x}"
                dataset_coarse[new_var] = (other_dims, arr_coarse[new_y-1, new_x-1])
                
    
    
    return dataset_coarse

In [12]:
metrics_dataset = xr.open_dataset("PPE_7_5deg_metrics.nc", engine="netcdf4")



weightsNames = get_weights_names(original_box_size)
default_metrics_dataset = metrics_dataset.sel(ens_idx="ctrl")

metricsWeights = default_metrics_dataset.isel(time=0,product=0)[weightsNames].to_array(dim="metricsNames").to_numpy()



In [13]:

coarse_dataset = coarsen_spatial_features(metrics_dataset,original_box_size, new_box_size, varPrefixes)

['SWCF_1_1', 'SWCF_1_2', 'SWCF_1_3', 'SWCF_1_4', 'SWCF_1_5', 'SWCF_1_6', 'SWCF_1_7', 'SWCF_1_8', 'SWCF_1_9', 'SWCF_1_10', 'SWCF_1_11', 'SWCF_1_12', 'SWCF_1_13', 'SWCF_1_14', 'SWCF_1_15', 'SWCF_1_16', 'SWCF_1_17', 'SWCF_1_18', 'SWCF_1_19', 'SWCF_1_20', 'SWCF_1_21', 'SWCF_1_22', 'SWCF_1_23', 'SWCF_1_24', 'SWCF_1_25', 'SWCF_1_26', 'SWCF_1_27', 'SWCF_1_28', 'SWCF_1_29', 'SWCF_1_30', 'SWCF_1_31', 'SWCF_1_32', 'SWCF_1_33', 'SWCF_1_34', 'SWCF_1_35', 'SWCF_1_36', 'SWCF_1_37', 'SWCF_1_38', 'SWCF_1_39', 'SWCF_1_40', 'SWCF_1_41', 'SWCF_1_42', 'SWCF_1_43', 'SWCF_1_44', 'SWCF_1_45', 'SWCF_1_46', 'SWCF_1_47', 'SWCF_1_48', 'SWCF_2_1', 'SWCF_2_2', 'SWCF_2_3', 'SWCF_2_4', 'SWCF_2_5', 'SWCF_2_6', 'SWCF_2_7', 'SWCF_2_8', 'SWCF_2_9', 'SWCF_2_10', 'SWCF_2_11', 'SWCF_2_12', 'SWCF_2_13', 'SWCF_2_14', 'SWCF_2_15', 'SWCF_2_16', 'SWCF_2_17', 'SWCF_2_18', 'SWCF_2_19', 'SWCF_2_20', 'SWCF_2_21', 'SWCF_2_22', 'SWCF_2_23', 'SWCF_2_24', 'SWCF_2_25', 'SWCF_2_26', 'SWCF_2_27', 'SWCF_2_28', 'SWCF_2_29', 'SWCF_2_30', 'SW

In [14]:
coarse_dataset.to_netcdf(f'PPE_{new_box_size}deg_metrics.nc')

In [10]:
coarse_dataset

<xarray.Dataset> Size: 950kB
Dimensions:    (ens_idx: 351, input_param: 14, product: 2, time: 1)
Coordinates:
  * ens_idx    (ens_idx) <U86 121kB 'ens/workdir.1/20230802.v3alpha02.F2010.p...
  * product    (product) <U3 24B 'mod' 'obs'
  * time       (time) <U3 12B 'ANN'
Dimensions without coordinates: input_param
Data variables: (12/217)
    params     (ens_idx, input_param) float32 20kB ...
    numb_1_1   float64 8B 0.006005
    numb_1_2   float64 8B 0.006005
    numb_1_3   float64 8B 0.006005
    numb_1_4   float64 8B 0.006005
    numb_1_5   float64 8B 0.006005
    ...         ...
    LWCF_6_7   (time, product, ens_idx) float64 6kB 18.58 17.99 ... 20.8 20.8
    LWCF_6_8   (time, product, ens_idx) float64 6kB 18.28 17.71 ... 21.8 21.8
    LWCF_6_9   (time, product, ens_idx) float64 6kB 20.2 20.32 ... 23.55 23.55
    LWCF_6_10  (time, product, ens_idx) float64 6kB 20.22 20.41 ... 22.99 22.99
    LWCF_6_11  (time, product, ens_idx) float64 6kB 15.35 15.55 ... 20.71 20.71
    LWCF_6_12  (time, product, ens_idx) float64 6kB 15.54 15.69 ... 19.51 19.51